**Links**<br>
Extensions API GitHub: https://github.com/SillyTavern/SillyTavern-extras/<br>
SillyTavern community Discord (support and discussion): https://discord.gg/sillytavern

In [ ]:
# ==========================================
# ⚙️ НАСТРОЙКИ МОДУЛЕЙ SILLYTAVERN EXTRAS
# ==========================================
secure = False
whisper_model = "base.en"
classification_model = "nateraw/bert-base-uncased-emotion"
summarization_model = "slauw87/bart_summarisation"
captioning_model = "Salesforce/blip-image-captioning-large"
sd_model = "ckpt/anything-v4.5-vae-swapped"

extras_enable_websearch = True
extras_enable_caption = True
extras_enable_classify = True
extras_enable_summarize = True
extras_enable_silero_tts = True
extras_enable_edge_tts = True
extras_enable_chromadb = True
extras_enable_whisper_stt = True
extras_enable_sd = True
# ==========================================

import subprocess
import secrets
import os
import glob

print("🚀 Запуск скрипта с защитным патчем для Silero TTS...")

# 1. Клонируем репозитории
if not os.path.exists('/SillyTavern-extras'):
    !git clone https://github.com/SillyTavern/SillyTavern-extras /SillyTavern-extras

%cd /SillyTavern-extras
if not os.path.exists('tts_samples'):
    !git clone https://github.com/Cohee1207/tts_samples

# 2. Ставим свежий pip и Pillow
!pip install --upgrade pip setuptools wheel
!pip install Pillow>=10.0.0

# 3. Устанавливаем зависимости
!pip install -r requirements.txt

# 4. Совместимость transformers
!pip install "transformers<5.0.0" accelerate

# ==========================================
# 🩹 ТРОЙНОЙ ПАТЧ ДЛЯ SILERO TTS (Torchaudio + Raise + 404 Fix)
# ==========================================
print("🩹 Применяем жесткий патч для Silero TTS...")
for path in glob.glob('/usr/local/lib/python*/site-packages/silero_api_server/tts.py') + glob.glob('/usr/local/lib/python*/dist-packages/silero_api_server/tts.py'):
    with open(path, 'r') as f:
        content = f.read()

    # Патч 1: Убираем сломанный torchaudio backend
    content = content.replace('torchaudio.set_audio_backend("soundfile")', '# torchaudio.set_audio_backend("soundfile")')

    # Патч 2 & 3: Перехватываем метод list_languages, чтобы он не делал мертвые запросы и не падал
    old_list_langs = """    def list_languages(self):
        response = requests.get("https://raw.githubusercontent.com/snakers4/silero-models/master/models.yml") # пример или подобный URL
        if response.status_code != 200:
            raise f"Failed to get languages: {response.status_code}\""""

    # Заменяем проблемный кусок на заглушку, которая возвращает базовые языки (ru, en)
    # Вместо точного поиска заменим весь метод list_languages на безопасный
    if "def list_languages(self):" in content:
        # Находим и заменяем логику метода
        target_snippet = """    def list_languages(self):
        return {"ru": ["v3_1_ru"], "en": ["v3_en"]}"""
        # Если оригинальный метод выглядит иначе, подменим его через поиск границ
        # Сделаем проще: заменим строку вызова на безопасную заглушку внутри класса
        content = content.replace(
            "self.langs = self.list_languages()",
            "self.langs = {'ru': ['v3_1_ru'], 'en': ['v3_en']}"
        )

    with open(path, 'w') as f:
        f.write(content)
    print(f"✅ Файл {path} успешно пропатчен от 404!")
# ==========================================

# 5. Качаем туннель Cloudflare
!wget -nc https://github.com/cloudflare/cloudflared/releases/download/2023.5.0/cloudflared-linux-amd64 -O /tmp/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared-linux-amd64

# 6. Собираем параметры и модули
params = ['--share', '--cuda']
if secure:
    params.append('--secure')

modules = []
if extras_enable_caption: modules.append('caption')
if extras_enable_summarize: modules.append('summarize')
if extras_enable_classify: modules.append('classify')
if extras_enable_sd: modules.append('sd')
if extras_enable_silero_tts: modules.append('silero-tts')
if extras_enable_edge_tts: modules.append('edge-tts')
if extras_enable_chromadb: modules.append('chromadb')
if extras_enable_whisper_stt:
    modules.append('whisper-stt')
    params.append(f'--stt-whisper-model-path={whisper_model}')

if extras_enable_websearch:
    modules.append('websearch')
    !apt-get update && apt-get install -y chromium-chromedriver

params.append(f'--classification-model={classification_model}')
params.append(f'--summarization-model={summarization_model}')
params.append(f'--captioning-model={captioning_model}')
params.append(f'--sd-model={sd_model}')
params.append(f'--enable-modules={",".join(modules)}')

# 7. Генерируем ключ
api_key = secrets.token_hex(5)
with open('./api_key.txt', 'w') as f:
    f.write(api_key)
print(f"🔑 Сгенерирован API Key: {api_key}")

# 8. Запускаем сервер
cmd = f"python server.py {' '.join(params)}"
print(f"📡 Команда запуска: {cmd}\n")

extras_process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd='/SillyTavern-extras', shell=True)
print('processId:', extras_process.pid)

while True:
    line = extras_process.stdout.readline().decode().strip()
    if line:
        print(line)

In [ ]:
#@title <-- Tap this if you run on Mobile { display-mode: "form" }
#Taken from KoboldAI colab
%%html
<b>Press play on the audio player to keep the tab alive. (Uses only 13MB of data)</b><br/>
<audio src="https://henk.tech/colabkobold/silence.m4a" controls>

In [ ]:
# ==========================================
# ⚙️ НАСТРОЙКИ МОДУЛЕЙ SILLYTAVERN EXTRAS
# ==========================================
secure = False                    # Генерация защищенного ключа
whisper_model = "base.en"         # Модель для распознавания речи
classification_model = "nateraw/bert-base-uncased-emotion"
summarization_model = "slauw87/bart_summarisation"
captioning_model = "Salesforce/blip-image-captioning-large"
sd_model = "ckpt/anything-v4.5-vae-swapped"

# Включение/выключение модулей (можешь выключить что-то ненужное, чтобы ускорить запуск)
extras_enable_websearch = True
extras_enable_caption = True
extras_enable_classify = True
extras_enable_summarize = True
extras_enable_silero_tts = True
extras_enable_edge_tts = True
extras_enable_chromadb = True
extras_enable_whisper_stt = True
extras_enable_sd = True           # Ставит Stable Diffusion (требует около 4ГБ на скачивание)
# ==========================================

import subprocess
import secrets
import os

print("🚀 Запуск чистой и исправленной установки SillyTavern Extras...")

# 1. Клонируем репозитории, если их еще нет
if not os.path.exists('/SillyTavern-extras'):
    !git clone https://github.com/SillyTavern/SillyTavern-extras /SillyTavern-extras

%cd /SillyTavern-extras
if not os.path.exists('tts_samples'):
    !git clone https://github.com/Cohee1207/tts_samples

# 2. Ставим свежий pip и современный Pillow (обход ошибки сборки на Python 3.13)
!pip install --upgrade pip setuptools wheel
!pip install Pillow>=10.0.0

# 3. Устанавливаем базовые требования из репозитория
!pip install -r requirements.txt

# 4. ГЛАВНЫЙ ФИКС ОКРУЖЕНИЯ:
# Возвращаем нормальный PyTorch с поддержкой GPU (CUDA 12.1) и ставим стабильный transformers
print("🛠️ Применяем исправления: настраиваем GPU и совместимость библиотек...")
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install "transformers<5.0.0" accelerate

# 5. Качаем туннель Cloudflare для создания публичной ссылки
!wget -nc https://github.com/cloudflare/cloudflared/releases/download/2023.5.0/cloudflared-linux-amd64 -O /tmp/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared-linux-amd64

# 6. Собираем параметры и список активных модулей
params = ['--share', '--cuda']
if secure:
    params.append('--secure')

modules = []
if extras_enable_caption: modules.append('caption')
if extras_enable_summarize: modules.append('summarize')
if extras_enable_classify: modules.append('classify')
if extras_enable_sd: modules.append('sd')
if extras_enable_silero_tts: modules.append('silero-tts')
if extras_enable_edge_tts: modules.append('edge-tts')
if extras_enable_chromadb: modules.append('chromadb')
if extras_enable_whisper_stt:
    modules.append('whisper-stt')
    params.append(f'--stt-whisper-model-path={whisper_model}')

if extras_enable_websearch:
    print("🌍 Подключаем зависимости для WebSearch...")
    modules.append('websearch')
    !apt-get update && apt-get install -y chromium-chromedriver

params.append(f'--classification-model={classification_model}')
params.append(f'--summarization-model={summarization_model}')
params.append(f'--captioning-model={captioning_model}')
params.append(f'--sd-model={sd_model}')
params.append(f'--enable-modules={",".join(modules)}')

# 7. Генерируем случайный API-ключ для безопасности
api_key = secrets.token_hex(5)
with open('./api_key.txt', 'w') as f:
    f.write(api_key)
print(f"🔑 Сгенерирован API Key: {api_key}")

# 8. Запускаем сервер в фоновом потоке и выводим логи
cmd = f"python server.py {' '.join(params)}"
print(f"📡 Команда запуска: {cmd}\n")

extras_process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd='/SillyTavern-extras', shell=True)
print('processId:', extras_process.pid)

while True:
    line = extras_process.stdout.readline().decode().strip()
    if line:
        print(line)

In [ ]:
# ==========================================
# ⚙️ НАСТРОЙКИ МОДУЛЕЙ SILLYTAVERN EXTRAS
# ==========================================
secure = False
whisper_model = "base.en"
classification_model = "nateraw/bert-base-uncased-emotion"
summarization_model = "slauw87/bart_summarisation"
captioning_model = "Salesforce/blip-image-captioning-large"
sd_model = "ckpt/anything-v4.5-vae-swapped"

extras_enable_websearch = True
extras_enable_caption = True
extras_enable_classify = True
extras_enable_summarize = True
extras_enable_silero_tts = True
extras_enable_edge_tts = True
extras_enable_chromadb = True
extras_enable_whisper_stt = True
extras_enable_sd = True
# ==========================================

import subprocess
import secrets
import os

print("🚀 Запуск чистого скрипта SillyTavern Extras (без лишних переустановок PyTorch)...")

# 1. Клонируем репозитории
if not os.path.exists('/SillyTavern-extras'):
    !git clone https://github.com/SillyTavern/SillyTavern-extras /SillyTavern-extras

%cd /SillyTavern-extras
if not os.path.exists('tts_samples'):
    !git clone https://github.com/Cohee1207/tts_samples

# 2. Ставим свежий pip и современный Pillow
!pip install --upgrade pip setuptools wheel
!pip install Pillow>=10.0.0

# 3. Устанавливаем зависимости из requirements.txt
!pip install -r requirements.txt

# 4. Гарантируем совместимость transformers (без сноса родного PyTorch Colab)
print("🛠️ Настраиваем совместимость библиотек...")
!pip install "transformers<5.0.0" accelerate

# 5. Качаем туннель Cloudflare
!wget -nc https://github.com/cloudflare/cloudflared/releases/download/2023.5.0/cloudflared-linux-amd64 -O /tmp/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared-linux-amd64

# 6. Собираем параметры и модули
params = ['--share', '--cuda']
if secure:
    params.append('--secure')

modules = []
if extras_enable_caption: modules.append('caption')
if extras_enable_summarize: modules.append('summarize')
if extras_enable_classify: modules.append('classify')
if extras_enable_sd: modules.append('sd')
if extras_enable_silero_tts: modules.append('silero-tts')
if extras_enable_edge_tts: modules.append('edge-tts')
if extras_enable_chromadb: modules.append('chromadb')
if extras_enable_whisper_stt:
    modules.append('whisper-stt')
    params.append(f'--stt-whisper-model-path={whisper_model}')

if extras_enable_websearch:
    print("🌍 Подключаем зависимости для WebSearch...")
    modules.append('websearch')
    !apt-get update && apt-get install -y chromium-chromedriver

params.append(f'--classification-model={classification_model}')
params.append(f'--summarization-model={summarization_model}')
params.append(f'--captioning-model={captioning_model}')
params.append(f'--sd-model={sd_model}')
params.append(f'--enable-modules={",".join(modules)}')

# 7. Генерируем ключ
api_key = secrets.token_hex(5)
with open('./api_key.txt', 'w') as f:
    f.write(api_key)
print(f"🔑 Сгенерирован API Key: {api_key}")

# 8. Запускаем сервер
cmd = f"python server.py {' '.join(params)}"
print(f"📡 Команда запуска: {cmd}\n")

extras_process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd='/SillyTavern-extras', shell=True)
print('processId:', extras_process.pid)

while True:
    line = extras_process.stdout.readline().decode().strip()
    if line:
        print(line)

In [ ]:
# Устанавливаем совместимую 4-ю версию transformers
!pip install "transformers<5.0.0" "torch" "accelerate"

In [ ]:
%cd /
!git clone https://github.com/SillyTavern/SillyTavern-extras
%cd /SillyTavern-extras
!git clone https://github.com/Cohee1207/tts_samples
!npm install -g localtunnel
%pip install -r requirements.txt
!wget https://github.com/cloudflare/cloudflared/releases/download/2023.5.0/cloudflared-linux-amd64 -O /tmp/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared-linux-amd64

if extras_enable_rvc:
  print("Installing RVC requirements")
  %pip install -r requirements-rvc.txt

# ==========================================
# 🛠️ ИСПРАВЛЕНИЕ ДЛЯ GPU И БИБЛИОТЕК:
# 1. Возвращаем PyTorch с поддержкой CUDA (никаких CPU!)
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# 2. Фиксим transformers и accelerate под стабильную версию
!pip install "transformers<5.0.0" accelerate
# ==========================================

# Generate a random API key
api_key = secrets.token_hex(5)
...

In [ ]:
#@markdown (RECOMMENDED) Generates an API key for you to use with the API
secure = False #@param {type:"boolean"}
#@markdown Allows to run SillyTavern Extras on CPU (use if you're out of daily GPU allowance)
use_cpu = False #@param {type:"boolean"}
#@markdown Allows to run Stable Diffusion pipeline on CPU (slow!)
use_sd_cpu = False #@param {type:"boolean"}
#@markdown ***
#@markdown Enables the WebSearch module
extras_enable_websearch = True #@param {type:"boolean"}
#@markdown ***
#@markdown Loads the image captioning module
extras_enable_caption = True #@param {type:"boolean"}
captioning_model = "Salesforce/blip-image-captioning-large" #@param [ "Salesforce/blip-image-captioning-large", "Salesforce/blip-image-captioning-base" ]
#@markdown * Salesforce/blip-image-captioning-large - good base model
#@markdown * Salesforce/blip-image-captioning-base - slightly faster but less accurate
#@markdown ***
#@markdown Loads the sentiment classification model
extras_enable_classify = True #@param {type:"boolean"}
classification_model = "nateraw/bert-base-uncased-emotion" #@param ["nateraw/bert-base-uncased-emotion", "joeddav/distilbert-base-uncased-go-emotions-student"]
#@markdown * nateraw/bert-base-uncased-emotion = 6 supported emotions<br>
#@markdown * joeddav/distilbert-base-uncased-go-emotions-student = 28 supported emotions
#@markdown ***
#@markdown Loads the story summarization module
extras_enable_summarize = True #@param {type:"boolean"}
summarization_model = "slauw87/bart_summarisation" #@param [ "slauw87/bart_summarisation", "Qiliang/bart-large-cnn-samsum-ChatGPT_v3", "Qiliang/bart-large-cnn-samsum-ElectrifAi_v10", "distilbart-xsum-12-3" ]
#@markdown * slauw87/bart_summarisation - general purpose summarization model
#@markdown * Qiliang/bart-large-cnn-samsum-ChatGPT_v3 - summarization model optimized for chats
#@markdown * Qiliang/bart-large-cnn-samsum-ElectrifAi_v10 - nice results so far, but still being evaluated
#@markdown * distilbart-xsum-12-3 - faster, but pretty basic alternative
#@markdown ***
#@markdown Enables Silero text-to-speech module
extras_enable_silero_tts = True #@param {type:"boolean"}
#@markdown Enables Microsoft Edge text-to-speech module
extras_enable_edge_tts = True #@param {type:"boolean"}
#@markdown Enables RVC module
extras_enable_rvc = False #@param {type:"boolean"}
#@markdown ***
#@markdown Enables Whisper speech recognition module
extras_enable_whisper_stt = True #@param {type:"boolean"}
whisper_model = "base.en" #@param [ "tiny.en", "base.en", "small.en", "medium.en", "tiny", "base", "small", "medium", "large" ]
#@markdown There are five model sizes, four with English-only versions, offering speed and accuracy tradeoffs.
#@markdown The .en models for English-only applications tend to perform better, especially for the tiny.en and base.en models.
#@markdown ***
#@markdown Enables SD picture generation
extras_enable_sd = True #@param {type:"boolean"}
sd_model = "ckpt/anything-v4.5-vae-swapped" #@param [ "ckpt/anything-v4.5-vae-swapped", "hakurei/waifu-diffusion", "philz1337/clarity", "prompthero/openjourney", "ckpt/sd15", "stabilityai/stable-diffusion-2-1-base" ]
#@markdown * ckpt/anything-v4.5-vae-swapped - anime style model
#@markdown * hakurei/waifu-diffusion - anime style model
#@markdown * philz1337/clarity - realistic style model
#@markdown * prompthero/openjourney - midjourney style model
#@markdown * ckpt/sd15 - base SD 1.5
#@markdown * stabilityai/stable-diffusion-2-1-base - base SD 2.1
#@markdown ***
#@markdown Enables ChromaDB module
extras_enable_chromadb = True #@param {type:"boolean"}

import subprocess
import secrets

# ---
# SillyTavern extras
extras_url = '(disabled)'
params = []
if use_cpu:
    params.append('--cpu')
if use_sd_cpu:
    params.append('--sd-cpu')
if secure:
    params.append('--secure')
params.append('--share')
modules = []

if extras_enable_caption:
  modules.append('caption')
if extras_enable_summarize:
  modules.append('summarize')
if extras_enable_classify:
  modules.append('classify')
if extras_enable_sd:
  modules.append('sd')
if extras_enable_silero_tts:
  modules.append('silero-tts')
if extras_enable_edge_tts:
  modules.append('edge-tts')
if extras_enable_chromadb:
  modules.append('chromadb')
if extras_enable_whisper_stt:
  modules.append('whisper-stt')
  params.append(f'--stt-whisper-model-path={whisper_model}')
if extras_enable_rvc:
  modules.append('rvc')
  params.append('--max-content-length=2000')
  params.append('--rvc-save-file')


if extras_enable_websearch:
    print("Enabling WebSearch module")
    modules.append('websearch')
    !apt update
    !apt install -y chromium-chromedriver

params.append(f'--classification-model={classification_model}')
params.append(f'--summarization-model={summarization_model}')
params.append(f'--captioning-model={captioning_model}')
params.append(f'--sd-model={sd_model}')
params.append(f'--enable-modules={",".join(modules)}')


%cd /
!git clone https://github.com/SillyTavern/SillyTavern-extras
%cd /SillyTavern-extras
!git clone https://github.com/Cohee1207/tts_samples
!npm install -g localtunnel
%pip install -r requirements.txt
!wget https://github.com/cloudflare/cloudflared/releases/download/2023.5.0/cloudflared-linux-amd64 -O /tmp/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared-linux-amd64

if extras_enable_rvc:
  print("Installing RVC requirements")
  %pip install -r requirements-rvc.txt

# ==========================================
# 🛠️ ИСПРАВЛЕНИЕ ДЛЯ GPU И БИБЛИОТЕК:
# 1. Возвращаем PyTorch с поддержкой CUDA (никаких CPU!)
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# 2. Фиксим transformers и accelerate под стабильную версию
!pip install "transformers<5.0.0" accelerate
# ==========================================

# Generate a random API key
api_key = secrets.token_hex(5)

# Write the API key to api_key.txt
with open('./api_key.txt', 'w') as f:
    f.write(api_key)
print(f"API Key generated: {api_key}")

cmd = f"python server.py {' '.join(params)}"
print(cmd)
extras_process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd='/SillyTavern-extras', shell=True)
print('processId:', extras_process.pid)
while True:
    line = extras_process.stdout.readline().decode().strip()
    if line != None and line != '':
        print(line)


In [ ]:
!python server.py --share --cuda --cuda-device 0 \
  --stt-whisper-model-path=base.en \
  --classification-model=nateraw/bert-base-uncased-emotion \
  --summarization-model=slauw87/bart_summarisation \
  --captioning-model=Salesforce/blip-image-captioning-large \
  --sd-model=ckpt/anything-v4.5-vae-swapped \
  --enable-modules=caption,summarize,classify,sd,silero-tts,edge-tts,chromadb,whisper-stt,websearch

In [ ]:
!python server.py --share --cuda --cuda-device 0 --stt-whisper-model-path=base.en --classification-model=nateraw/bert-base-uncased-emotion --summarization-model=slauw87/bart_summarisation --captioning-model=Salesforce/blip-image-captioning-large --sd-model=ckpt/anything-v4.5-vae-swapped --enable-modules=caption,summarize,classify,sd,silero-tts,edge-tts,chromadb,whisper-stt,websearch

In [ ]:
!pip install --upgrade pip setuptools wheel
!pip install --only-binary=:all: Pillow

In [ ]:

!apt-get update && apt-get install -y libjpeg-dev zlib1g-dev
!pip install --upgrade pip
%cd /content/SillyTavern-extras
!pip install -r requirements.txt
# 1. Переходим в папку extras (проверь, как она у тебя называется, обычно /content/SillyTavern-extras или просто /SillyTavern-extras)
%cd /content/SillyTavern-extras

# 2. Обновляем pip на всякий случай
!pip install --upgrade pip

# 3. Устанавливаем все зависимости из файла requirements.txt
!pip install -r requirements.txt